# Tech Challenge FIAP Fase 4 — Execução Completa no Colab

Este notebook executa a **fusão multimodal** de uma só vez:

1. Baixa os dados reais do eICU Demo (PhysioNet).
2. Usa um vídeo de fisioterapia (padrão do repo ou um específico do REHAB24-6).
3. Roda `main.py`, que orquestra o módulo clínico, o módulo de vídeo e a fusão.
4. Exibe o relatório final consolidado.

> Ambiente de execução: **CPU é suficiente**. GPU não acelera significativamente o MediaPipe.

## 1. Clonar repositório e instalar dependências

In [1]:
import os

# Configuracao de branch: use 'main' para versao final ou 'feat/fusao-multimodal'
# para testar as ultimas alteracoes durante o desenvolvimento.
BRANCH = os.environ.get('NOTEBOOK_BRANCH', 'main')
print(f'Clonando/Puxando branch: {BRANCH}')

PROJETO = '/content/Tech-Challenge-FIAP-FASE4'
if not os.path.exists(PROJETO):
    !git clone -b {BRANCH} https://github.com/elbergaliza/Tech-Challenge-FIAP-FASE4.git {PROJETO}
else:
    !cd {PROJETO} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

!pip install -q -r {PROJETO}/requirements.txt
!pip install -q -e {PROJETO}

# No Colab há conflito mediapipe x tensorflow pré-instalado
!pip uninstall -y -q tensorflow tensorflow-cpu 2>/dev/null

%cd {PROJETO}
print('Pronto!')

Clonando/Puxando branch: main
Cloning into '/content/Tech-Challenge-FIAP-FASE4'...
remote: Enumerating objects: 560, done.
remote: Counting objects: 100% (490/490), done.
remote: Compressing objects: 100% (305/305), done.
remote: Total 560 (delta 184), reused 449 (delta 154), pack-reused 70 (from 3)
Receiving objects: 100% (560/560), 131.64 MiB | 17.35 MiB/s, done.
Resolving deltas: 100% (184/184), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Baixar dados reais do eICU Demo (PhysioNet)

O eICU Collaborative Research Database Demo é público e não exige login para
os arquivos de demo.



In [2]:
import os
import urllib.request

EICU_DIR = f'{PROJETO}/eicu-anomaly-detection/modulo_anomalias/data/raw'
os.makedirs(EICU_DIR, exist_ok=True)

base_url = 'https://physionet.org/files/eicu-crd-demo/2.0.1/'
arquivos = ['vitalPeriodic.csv.gz', 'lab.csv.gz', 'medication.csv.gz']

for arq in arquivos:
    destino = os.path.join(EICU_DIR, arq)
    if os.path.exists(destino):
        print(f'{arq} ja existe.')
        continue
    url = base_url + arq
    print(f'Baixando {arq}...')
    try:
        urllib.request.urlretrieve(url, destino)
        print(f'  OK: {destino} ({os.path.getsize(destino) / 1024 / 1024:.1f} MB)')
    except Exception as e:
        print(f'  ERRO: {e}')
        raise

print('Dados eICU prontos.')

Baixando vitalPeriodic.csv.gz...
  OK: /content/Tech-Challenge-FIAP-FASE4/eicu-anomaly-detection/modulo_anomalias/data/raw/vitalPeriodic.csv.gz (18.5 MB)
Baixando lab.csv.gz...
  OK: /content/Tech-Challenge-FIAP-FASE4/eicu-anomaly-detection/modulo_anomalias/data/raw/lab.csv.gz (5.5 MB)
Baixando medication.csv.gz...
  OK: /content/Tech-Challenge-FIAP-FASE4/eicu-anomaly-detection/modulo_anomalias/data/raw/medication.csv.gz (1.4 MB)
Dados eICU prontos.


## 3. Definir o vídeo para o módulo de vídeo

O notebook usa a variável de ambiente `REHAB_VIDEO_ID` para escolher
qual vídeo do dataset REHAB24-6 será analisado.

- Valor padrão: `PM_029` (vídeo já incluso no repositório).
- Para usar outro vídeo, defina `REHAB_VIDEO_ID` antes de executar a célula
  abaixo (ex.: `PM_038`).
- IDs disponíveis para Camera17 no Ex6: PM_008, PM_022, PM_029, PM_038,
  PM_043, PM_105, PM_113, PM_118, PM_126.

```python
import os
os.environ['REHAB_VIDEO_ID'] = 'PM_029'  # altere aqui se desejar
```

In [3]:
import os

# Le a variavel de ambiente REHAB_VIDEO_ID.
# Se nao estiver definida, usa 'PM_029' (video padrao do repositorio).
VIDEO_ID = os.environ.get('REHAB_VIDEO_ID', 'PM_029')

VIDEO_REAL = f'{PROJETO}/modulo_video/data/exemplos/{VIDEO_ID}-Camera17-30fps.mp4'
VIDEO_PATIENT_ID = VIDEO_ID
print(f'Video selecionado: {VIDEO_REAL}')
print(f'Existe: {os.path.exists(VIDEO_REAL)}')

Video selecionado: /content/Tech-Challenge-FIAP-FASE4/modulo_video/data/exemplos/PM_029-Camera17-30fps.mp4
Existe: False


### Opção avançada — baixar outro vídeo do REHAB24-6

Se `REHAB_VIDEO_ID` aponta para um vídeo que não está em
`modulo_video/data/exemplos/`, execute a célula abaixo para baixar apenas
esse vídeo do dataset REHAB24-6 (Zenodo). O download do arquivo compactado
é de ~2.6 GB, então pode demorar alguns minutos. Se o vídeo já existir,
a célula pula automaticamente.

In [4]:
import os
import urllib.request
import zipfile

VIDEO_ID = os.environ.get('REHAB_VIDEO_ID', 'PM_029')

VIDEO_LOCAL = f'{PROJETO}/modulo_video/data/exemplos/{VIDEO_ID}-Camera17-30fps.mp4'
if os.path.exists(VIDEO_LOCAL):
    print(f'Video {VIDEO_ID} ja existe no repositorio. Nenhum download necessario.')
    VIDEO_REAL = VIDEO_LOCAL
    VIDEO_PATIENT_ID = VIDEO_ID
else:
    REHAB_DIR = '/content/REHAB24-6'
    os.makedirs(REHAB_DIR, exist_ok=True)

    videos_zip = '/tmp/rehab_videos.zip'
    video_rehab = f'{REHAB_DIR}/videos/Ex6/{VIDEO_ID}-Camera17-30fps.mp4'
    if not os.path.exists(video_rehab):
        print(f'Baixando videos.zip do REHAB24-6 (2.6 GB, pode levar)...')
        urllib.request.urlretrieve('https://zenodo.org/records/13305826/files/videos.zip', videos_zip)
        print(f'Descompactando apenas o video {VIDEO_ID}...')
        with zipfile.ZipFile(videos_zip) as z:
            alvo = f'Ex6/{VIDEO_ID}-Camera17-30fps.mp4'
            z.extract(alvo, f'{REHAB_DIR}/videos')
        print('OK')

    VIDEO_REAL = video_rehab
    VIDEO_PATIENT_ID = VIDEO_ID

print(f'Video selecionado: {VIDEO_REAL}')
print(f'Existe: {os.path.exists(VIDEO_REAL)}')

Baixando videos.zip do REHAB24-6 (2.6 GB, pode levar)...
Descompactando apenas o video PM_029...
OK
Video selecionado: /content/REHAB24-6/videos/Ex6/PM_029-Camera17-30fps.mp4
Existe: True


## 4. Verificar dados antes da fusão

In [5]:
import os
print(f'VIDEO_REAL: {VIDEO_REAL}')
print(f'VIDEO_REAL existe: {os.path.exists(VIDEO_REAL)}')
print(f'EICU_DIR: {EICU_DIR}')
print(f'EICU_DIR existe: {os.path.exists(EICU_DIR)}')
for arq in ['vitalPeriodic.csv.gz', 'lab.csv.gz', 'medication.csv.gz']:
    print(f'  {arq}: {os.path.exists(os.path.join(EICU_DIR, arq))}')

VIDEO_REAL: /content/REHAB24-6/videos/Ex6/PM_029-Camera17-30fps.mp4
VIDEO_REAL existe: True
EICU_DIR: /content/Tech-Challenge-FIAP-FASE4/eicu-anomaly-detection/modulo_anomalias/data/raw
EICU_DIR existe: True
  vitalPeriodic.csv.gz: True
  lab.csv.gz: True
  medication.csv.gz: True


## 5. Executar a fusão multimodal

Esta é a célula principal. Ela roda `main.py`, que orquestra o módulo clínico,
o módulo de vídeo e a fusão dos alertas em um único relatório JSON.

In [6]:
import os
os.makedirs(f'{PROJETO}/outputs', exist_ok=True)

!python main.py \
  --video {VIDEO_REAL} \
  --eicu-data {EICU_DIR} \
  --video-patient-id {VIDEO_PATIENT_ID} \
  --sem-objetos \
  --saida {PROJETO}/outputs/final_multimodal_report.json

 Fusão Multimodal — Tech Challenge FIAP Fase 4

[main] Dados eICU: /content/Tech-Challenge-FIAP-FASE4/eicu-anomaly-detection/modulo_anomalias/data/raw
[main] Vídeo: /content/REHAB24-6/videos/Ex6/PM_029-Camera17-30fps.mp4
[main] Clinical patient id: 141761
[main] Video patient id: PM_029
[fusion] Executando adapter: ClinicalAdapter
[clinical] data_dir=/content/Tech-Challenge-FIAP-FASE4/eicu-anomaly-detection/modulo_anomalias/data/raw, patient_id=141761
[clinical] vitalPeriodic carregado: 1634960 linhas, 19 colunas
[clinical] features criadas: 2375 pacientes, 53 colunas
[clinical] modo=batch, train=2375, predict=2375
[clinical] predicoes: 2375 linhas, anomalias=238
[clinical] alertas gerados: 0
[fusion]   -> 0 alerta(s) gerado(s)
[fusion] Executando adapter: VideoAdapter
[1/5] Extraindo pose de: /content/REHAB24-6/videos/Ex6/PM_029-Camera17-30fps.mp4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1784507009.635525    2030 inference_feedback_manager.cc:114] Feedback m

## 6. Visualizar relatório final e resumo por módulo

In [7]:
import json
with open(f'{PROJETO}/outputs/final_multimodal_report.json') as f:
    relatorio = json.load(f)

print('RESUMO POR MODULO:')
por_modulo = {}
for alerta in relatorio.get('alertas', []):
    modulo = alerta.get('modulo')
    por_modulo.setdefault(modulo, 0)
    por_modulo[modulo] += 1
for modulo, qtd in por_modulo.items():
    print(f'  {modulo}: {qtd} alerta(s)')

print()
print(json.dumps(relatorio, ensure_ascii=False, indent=2))

RESUMO POR MODULO:
  video_fisioterapia: 1 alerta(s)

{
  "gerado_em": "2026-07-20T00:28:01.982138",
  "resumo": {
    "total_alertas": 1,
    "score_medio": 0.0,
    "nivel_mais_critico": "baixo",
    "modulos_analisados": [
      "video_fisioterapia"
    ],
    "modulos_com_alerta": [
      "video_fisioterapia"
    ],
    "recomendacao_geral": "Continuar monitoramento preventivo."
  },
  "alertas": [
    {
      "module_id": "PM_029",
      "modulo": "video_fisioterapia",
      "tipo_anomalia": "movimento",
      "score_risco": 0.0,
      "nivel_risco": "baixo",
      "descricao": "Nenhum desvio significativo de postura ou marcha.",
      "recomendacao": "Nenhum desvio relevante detectado. Manter acompanhamento de rotina."
    }
  ]
}


## 7. (opcional) Baixar resultados

In [8]:
from google.colab import files
files.download(f'{PROJETO}/outputs/final_multimodal_report.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## 8. (opcional) Calibração com REHAB24-6

Esta seção é opcional. Ela baixa o dataset completo do Zenodo, calibra
limiares para agachamento (Ex6) e mostra a comparação antes/depois.
Use `--max 4` para teste rápido, ou remova para processar todas as gravações.

In [9]:
import urllib.request
import zipfile

REHAB_DIR = '/content/REHAB24-6'
os.makedirs(REHAB_DIR, exist_ok=True)

if not os.path.exists(f'{REHAB_DIR}/videos/Ex6'):
    print('Baixando videos.zip (2.6 GB)...')
    urllib.request.urlretrieve('https://zenodo.org/records/13305826/files/videos.zip', '/tmp/rehab_videos.zip')
    print('Descompactando...')
    with zipfile.ZipFile('/tmp/rehab_videos.zip') as z:
        z.extractall(f'{REHAB_DIR}/videos')
    print('OK')

if not os.path.exists(f'{REHAB_DIR}/Segmentation.csv'):
    print('Baixando Segmentation.csv...')
    urllib.request.urlretrieve('https://zenodo.org/records/13305826/files/Segmentation.csv', f'{REHAB_DIR}/Segmentation.csv')
    print('OK')

print('Dataset pronto.')

Baixando Segmentation.csv...
OK
Dataset pronto.


In [10]:
# Calibrar com apenas 4 gravações (rápido) ou remover --max 4 para todas
!python -m modulo_video.calibrar --raiz {REHAB_DIR} --camera Camera17 --exercise 6 --pular-frames 5 --max 4 --salvar-limiares modulo_video/data/saida/limiares_calibrados.json

[1/4] PM_008: SEM VIDEO casado, pulando.
[2/4] PM_022: SEM VIDEO casado, pulando.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1784507285.253335    3145 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784507285.344641    3145 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
[3/4] PM_029: 20 repeticoes medidas (PM_029-Camera17-30fps.mp4).
[4/4] PM_038: SEM VIDEO casado, pulando.

CSV salvo em: data/saida/calibracao.csv

===== COMPARATIVO CORRETO x INCORRET

In [11]:
with open(f'{PROJETO}/modulo_video/data/saida/limiares_calibrados.json') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))

{
  "inclinacao_tronco_graus": 4.4189,
  "assimetria_marcha": 0.3563,
  "instabilidade_lateral": 0.0098,
  "velocidade_min": 0.067,
  "velocidade_max": 0.0838
}


In [12]:
# Comparar alerta com limiares calibrados
!python -m modulo_video.comparar_video --video {VIDEO_REAL} --limiares modulo_video/data/saida/limiares_calibrados.json --sem-objetos

[1/5] Extraindo pose de: /content/REHAB24-6/videos/Ex6/PM_029-Camera17-30fps.mp4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1784507422.267223    3716 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784507422.321125    3716 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
      3090 frames com pessoa detectada.
[2/5] Calculando metricas biomecanicas...
      - inclinacao_tronco_graus: 2.8276588381428214
      - assimetria_marcha: 0.17054181539282495
    